In [1]:
import sys
import os
import pandas as pd

src_path = os.path.abspath(os.path.join(os.path.dirname("__file__"), '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from dense_index_bge import DenseIndex
from sparse_index import SparseIndex

In [2]:
court_consideration_df = pd.read_csv("../data/court_considerations.csv")
court_consideration_d = dict(zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist()))

law_df = pd.read_csv("../data/laws_de.csv")
law_d = dict(zip(law_df['citation'].tolist(), law_df['text'].tolist()))

court_doc = [{'citation':citation, 'text':text} for citation,text in zip(court_consideration_df['citation'].tolist(), court_consideration_df['text'].tolist())]
law_doc = [{'citation':citation, 'text':text} for citation,text in zip(law_df['citation'].tolist(), law_df['text'].tolist())]

test_df = pd.read_csv("../data/test_rewrite_001.csv")
_d = {}
for _, row in test_df.iterrows():
    if row['query_id'] not in _d:
        _d[row['query_id']] = [row['query']]
    else:
        _d[row['query_id']].append(row['query'])
test_dict = {k: v for k, v in sorted(_d.items())}

test_df = pd.read_csv("../data/test_rewrite_001.csv")

print("data loaded.")

data loaded.


In [3]:
from FlagEmbedding import FlagReranker, BGEM3FlagModel

model = BGEM3FlagModel('/root/.cache/modelscope/hub/models/BAAI/bge-m3', use_fp16=True, show_progress_bar=False)

dense_index = DenseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)
# dense_index = DenseIndex(model, "../data/processed/_dense_court_removed_citation", court_doc)

sparse_index = SparseIndex(model, "/root/autodl-fs/bge-processed/_dense_sparse_court/", court_doc)

DenseIndex.embeddings:  (2107648, 1024)


In [4]:
reranker = FlagReranker('/root/.cache/modelscope/hub/models/BAAI/bge-reranker-v2-m3', use_fp16=True, normalize=True)
# reranker = FlagReranker('../ft_data/merged_reranker', use_fp16=True, normalize=True)

In [5]:
test_df = pd.read_csv("../data/test_rewrite_001.csv")
test_id_2_query_d = {}
for query_id, query in zip(test_df['query_id'], test_df['query']):
    test_id_2_query_d[query_id] = query

In [6]:
%load_ext autoreload

%autoreload 2

import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm
import bge_utils
import numpy as np

test_multi_question_df = pd.read_csv("../data/test_rewrite_002.csv")

recall_citations_l = []

query_id_2_query_list = defaultdict(list)

for query_id, query in zip(test_multi_question_df['query_id'], test_multi_question_df['query']):
    query_id_2_query_list[query_id].append(query)

for query_id, query in test_id_2_query_d.items():
    query_id_2_query_list[query_id].append(query) # 完整的问题在最后一个

query_id_2_recall_hits = {}

recall_hits_l = []

cc_repeat_count = defaultdict(int)

for query_id, query_list in query_id_2_query_list.items():
    
    _hits = []
    for q in query_list:
        hits1 = dense_index.search_with_score(q, top_k=100)
        for hit,_ in hits1:
            cc_repeat_count[hit['citation']] += 1
        hits2 = sparse_index.search_with_score(q, top_k=100)
        for hit,_ in hits2:
            cc_repeat_count[hit['citation']] += 1
        _hits = hits_utils.merge_hits_with_score_l_by_max(_hits,hits_utils.merge_hits_with_score_l_by_max(hits1, hits2))
    
    raw_query = query_list[-1]             

    all_hits = _hits.copy()
    
    print(f"[{query_id}] hits_merge.len:", len(all_hits))

    query_id_2_recall_hits[query_id] = all_hits.copy()

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


[test_001] hits_merge.len: 515
[test_002] hits_merge.len: 644
[test_003] hits_merge.len: 475
[test_004] hits_merge.len: 394
[test_005] hits_merge.len: 510
[test_006] hits_merge.len: 562
[test_007] hits_merge.len: 254
[test_008] hits_merge.len: 305
[test_009] hits_merge.len: 499
[test_010] hits_merge.len: 414
[test_011] hits_merge.len: 636
[test_012] hits_merge.len: 533
[test_013] hits_merge.len: 735
[test_014] hits_merge.len: 526
[test_015] hits_merge.len: 514
[test_016] hits_merge.len: 455
[test_017] hits_merge.len: 401
[test_018] hits_merge.len: 708
[test_019] hits_merge.len: 389
[test_020] hits_merge.len: 630
[test_021] hits_merge.len: 683
[test_022] hits_merge.len: 401
[test_023] hits_merge.len: 441
[test_024] hits_merge.len: 662
[test_025] hits_merge.len: 527
[test_026] hits_merge.len: 391
[test_027] hits_merge.len: 371
[test_028] hits_merge.len: 388
[test_029] hits_merge.len: 572
[test_030] hits_merge.len: 470
[test_031] hits_merge.len: 339
[test_032] hits_merge.len: 555
[test_03

In [8]:
import numpy as np
import math
from collections import defaultdict
import lightgbm_utils
import lightgbm

def normalize_and_aggregate_lightgbm(query_id, cc_results: list[dict]) -> dict[str, float]:
    data_list = lightgbm_utils.reranked_hits_to_json(query_id, cc_results)
    query_data = convert_to_query_data(data_list)
    X, y, group = build_lgb_dataset(query_data)

    model = lightgbm.Booster(model_file='rank_model.txt')
    scores = booster.predict(X_test)

In [ ]:
%load_ext autoreload

%autoreload 2

import metric_utils
import citation_utils
import hits_utils
import reranker_utils
import rrf
from collections import defaultdict
from tqdm.notebook import tqdm

from openai import OpenAI
import deepseek_utils
import bge_utils
import evidence_utils
import numpy as np
import math
import os
import os.path
import pickle

query_id_l = []
result_citation_l = []

for idx, (query_id, query_list) in tqdm(enumerate(query_id_2_query_list.items()), total=len(query_id_2_query_list)):

    recall_hits = query_id_2_recall_hits[query_id]

    print("===>",query_id, ", recall_hits.len:", len(recall_hits))

    raw_query = query_list[-1]

    cc_with_score_l_raw = reranker_utils.rerank_by_batch_chunked2(reranker, raw_query, [hit for hit,_ in recall_hits])

    print("===>",query_id, ", cc_with_score_l_raw.len:", len(cc_with_score_l_raw))

    citation_scores = normalize_and_aggregate_lightgbm(query_id, cc_with_score_l_raw)

    sorted_citation_l = sorted([(citation,score) for citation, score in citation_scores.items()], key=lambda x:x[1], reverse=True)

    sorted_citation_l = [(citation,score) for citation,score in sorted_citation_l if citation in court_consideration_d or citation in law_d]

    # 将citation合并
    query_result = [citation for citation, _ in sorted_citation_l]
    
    query_result2 = [citation for citation in query_result if citation in court_consideration_d or citation in law_d]
    
    print("query_result2:", len(query_result2))
    
    result_citation_l.append(';'.join(query_result2))
    query_id_l.append(query_id)
    
    print(f"{query_id} query_result2.len:", len(query_result2))


result_df = pd.DataFrame({'query_id':query_id_l, 'predicted_citations':result_citation_l})
result_df.to_csv("../output/submission_all.csv", index=False)
    


In [17]:
limit=19
predicted_citation_l = []
sub_df = pd.read_csv("../output/submission_all.csv")
for query_id, preds in zip(sub_df['query_id'].tolist(), sub_df['predicted_citations'].tolist()):
    citation_l = preds.split(";")
    predicted_citation_l.append(';'.join(citation_l[:limit]))

new_sub_df = pd.DataFrame({'query_id':sub_df['query_id'].tolist(), 'predicted_citations':predicted_citation_l})
new_sub_df.to_csv("../output/submission2.csv", index=False)

In [12]:
limit=30
cc_limit=10
predicted_citation_l = []
sub_df = pd.read_csv("../output/submission_all.csv")
for query_id, preds in zip(sub_df['query_id'].tolist(), sub_df['predicted_citations'].tolist()):
    cc_l = []
    law_l = []
    
    pred_l = preds.split(';')
    for pred in pred_l:
        if pred in court_consideration_d:
            if len(cc_l) >= cc_limit:
                pass
            else:
                cc_l.append(pred)
    for pred in pred_l:
        if limit <= len(cc_l) + len(law_l):
                break
        if pred in law_d:
            law_l.append(pred)

    citation_l = cc_l.copy()
    citation_l.extend(law_l)

    predicted_citation_l.append(';'.join(citation_l[:limit]))

new_sub_df = pd.DataFrame({'query_id':sub_df['query_id'].tolist(), 'predicted_citations':predicted_citation_l})
new_sub_df.to_csv("../output/submission2.csv", index=False)